In [0]:
with state_totals as (
  select state_abbr,
         sum(cast(replace(`GHG emissions mtons CO2e`, ',', '') as double)) as total_emissions
  from emissions_data
  group by state_abbr
),
country_total as (
  select sum(total_emissions) as country_emissions
  from state_totals
),
top_10_states as (
  select state_abbr,
         total_emissions,
         round((total_emissions / country_emissions) * 100, 2) as pct_of_country
  from state_totals
  cross join country_total
  order by total_emissions desc
  limit 10
)
select state_abbr,
       total_emissions,
       pct_of_country,
       round(sum(pct_of_country) over (order by total_emissions desc rows between unbounded preceding and current row), 2) as cumulative_pct
from top_10_states
order by total_emissions desc
       
